In [5]:
"""
Script de Entrenamiento, GridSearch CV y Tracking con MLflow (train.py)
Materia: Gesti3n de Proyectos de Inteligencia Artificial
Actividad: 5
Alumno: Manuel de Jesus Ramiro Velazquez Tapia
Matr1cula: AL07183522
Docente: Luis Ariel Vazquez Piña
"""

import os
import sys
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

# Intentar importar MLflow
try:
    import mlflow
    import mlflow.sklearn
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
    print("[WARN] MLflow no instalado en este entorno")

PATH_LIMPIO = "datos/datos_limp/breast_cancer_clean.csv"

def cargar_o_preparar_dataset():
    """Carga directamente el dataset limpio existente."""
    if not os.path.exists(PATH_LIMPIO):
        raise FileNotFoundError(f"No se encontro el archivo de datos limpio en: '{PATH_LIMPIO}'")

    print(f"[INFO] Cargando dataset limpio existente desde: '{PATH_LIMPIO}'")
    df = pd.read_csv(PATH_LIMPIO)
    return df

def entrenar_y_registrar():
    # 1. Cargar el dataset procesado existente
    df = cargar_o_preparar_dataset()
    X = df.drop(columns=['target'])
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    if HAS_MLFLOW:
        mlflow.set_experiment("Breast_Cancer_Classification_Act5")

    # ----------------------------------------------------
    # PIPELINE 1: REGRESI3N LOG1STICA
    # ----------------------------------------------------
    param_grid_lr = {'C': [0.1, 1.0, 10.0], 'penalty': ['l2'], 'solver': ['lbfgs']}
    grid_lr = GridSearchCV(LogisticRegression(max_iter=1000), param_grid_lr, cv=cv, scoring='f1')
    grid_lr.fit(X_train, y_train)

    best_lr = grid_lr.best_estimator_
    y_pred_lr = best_lr.predict(X_test)
    y_proba_lr = best_lr.predict_proba(X_test)[:, 1]

    acc_lr = accuracy_score(y_test, y_pred_lr)
    prec_lr = precision_score(y_test, y_pred_lr)
    rec_lr = recall_score(y_test, y_pred_lr)
    f1_lr = f1_score(y_test, y_pred_lr)
    auc_lr = roc_auc_score(y_test, y_proba_lr)

    if HAS_MLFLOW:
        with mlflow.start_run(run_name="Logistic_Regression_GridSearch"):
            mlflow.log_params(grid_lr.best_params_)
            mlflow.log_metrics({"accuracy": acc_lr, "precision": prec_lr, "recall": rec_lr, "f1_score": f1_lr, "roc_auc": auc_lr})
            mlflow.sklearn.log_model(best_lr, "model_logistic_regression")

    print(f"[LR] Regresi3n Log1stica -3 Accuracy: {acc_lr:.4f}, Precision: {prec_lr:.4f}, Recall: {rec_lr:.4f}, F1: {f1_lr:.4f}, ROC-AUC: {auc_lr:.4f}")

    # ----------------------------------------------------
    # PIPELINE 2: RANDOM FOREST CLASSIFIER
    # ----------------------------------------------------
    param_grid_rf = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20], 'random_state': [42]}
    grid_rf = GridSearchCV(RandomForestClassifier(), param_grid_rf, cv=cv, scoring='f1')
    grid_rf.fit(X_train, y_train)

    best_rf = grid_rf.best_estimator_
    y_pred_rf = best_rf.predict(X_test)
    y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

    acc_rf = accuracy_score(y_test, y_pred_rf)
    prec_rf = precision_score(y_test, y_pred_rf)
    rec_rf = recall_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)
    auc_rf = roc_auc_score(y_test, y_proba_rf)

    if HAS_MLFLOW:
        with mlflow.start_run(run_name="Random_Forest_GridSearch"):
            mlflow.log_params(grid_rf.best_params_)
            mlflow.log_metrics({"accuracy": acc_rf, "precision": prec_rf, "recall": rec_rf, "f1_score": f1_rf, "roc_auc": auc_rf})
            mlflow.sklearn.log_model(best_rf, "model_random_forest")

    print(f"[RF] Random Forest Classifier -3 Accuracy: {acc_rf:.4f}, Precision: {prec_rf:.4f}, Recall: {rec_rf:.4f}, F1: {f1_rf:.4f}, ROC-AUC: {auc_rf:.4f}")

if __name__ == "__main__":
    entrenar_y_registrar()

[WARN] MLflow no instalado en este entorno
[INFO] Cargando dataset limpio existente desde: 'datos/datos_limp/breast_cancer_clean.csv'
[LR] Regresi3n Log1stica -3 Accuracy: 0.9860, Precision: 0.9889, Recall: 0.9889, F1: 0.9889, ROC-AUC: 0.9977
[RF] Random Forest Classifier -3 Accuracy: 0.9580, Precision: 0.9565, Recall: 0.9778, F1: 0.9670, ROC-AUC: 0.9949
